# Piezo Triangle Scan - Beam Position Analysis

Loads a CSV produced by `piezo_triangle_scan.py` (columns: sample, timestamp,
voltage_setpoint_V, voltage_readback_V, centroid_x_um, centroid_y_um,
gaussfit_center_x_um, gaussfit_center_y_um, gaussfit_rating_x, gaussfit_rating_y,
peak_intensity_counts, total_power_mW, points_per_ramp, n_cycles) and plots:

- voltage and **mirror tilt angle** vs. sample (time series)
- **driven-axis mirror angle vs. piezo voltage** (the mirror's transfer
  function), colored by sample order so any hysteresis between the rising and
  falling ramps of the triangle wave is visible
- a linear fit to estimate the driven-axis sensitivity in urad/V

The camera measures where the reflected beam lands, in microns. A mirror tilt
of angle θ steers the beam by 2θ, which over the mirror→detector distance
`DETECTOR_DISTANCE` (meters, set in the config cell) becomes a 2θ·L displacement
on the sensor, so every camera displacement is converted to a **mirror tilt
angle in microradians** via θ = d / (2L) (`um_to_mirror_urad`). All displacement
plots and sensitivity/hysteresis numbers below are in mirror-angle units.

Set **`OMIT_FIRST_PASS`** (config cell below) to drop each scan's initial rising
ramp - the piezo's first pass often shows warm-up/conditioning that isn't part of
the steady hysteresis loop. The ramp length comes from the `points_per_ramp`
column when present (newer scans) and is otherwise inferred from the
voltage-setpoint triangle. The toggle applies to both the single-scan and the
master-run sections.

The driven axis (X or Y) is read from the filename's `_axisX`/`_axisY` token.
All plots use the Gaussian-fit center (converted to mirror angle) rather than
the plain centroid. Everything is reported in the **piezo axis frame**: because
the camera is mounted rotated 90 deg, a piezo axis shows up on the *other* camera
axis (piezo X -> camera Y, piezo Y -> camera X), so each plot selects the
camera column that carries the driven axis's motion and labels it by the piezo
axis.

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

# new scans are written here by piezo_triangle_scan.py / piezo_master_scan.py
DATA_DIR = "scans"

# Distance from the steering mirror to the beam-profiler sensor (meters). Used to
# turn a beam displacement on the camera into the mirror tilt angle that caused
# it (see um_to_mirror_urad below), so all results are reported as mirror angle
# rather than raw camera displacement.
DETECTOR_DISTANCE = 0.42


def um_to_mirror_urad(disp_um):
    """Camera-plane beam displacement (um) -> mirror tilt angle (microradians).

    A mirror tilted by angle theta steers the reflected beam by 2*theta; over
    the mirror->detector distance L that beam lands d = 2*theta*L off-center on
    the camera, so theta = d / (2 L). With d in um and L in m the unit factors
    cancel (1e-6 m/um * 1e6 urad/rad = 1), so dividing the um displacement by
    2*L (meters) yields the mirror angle directly in microradians.
    """
    return disp_um / (2.0 * DETECTOR_DISTANCE)


def add_mirror_angle_cols(d):
    """Add a mirror-angle column (urad) beside every camera-displacement (_um)
    column, named with a `_urad` suffix, and return `d`. All downstream plots
    and fits use these angle columns instead of the raw displacement."""
    for col in [c for c in d.columns if c.endswith("_um")]:
        d[col[:-len("_um")] + "_urad"] = um_to_mirror_urad(d[col])
    return d


# Toggle: omit the first up-ramp ("first pass") of each scan from the analysis.
# The piezo's initial rising ramp often shows warm-up/conditioning behaviour that
# isn't representative of the steady hysteresis loop, so it can be dropped here.
# This applies to both the single-scan section and every sub-scan of a master run.
OMIT_FIRST_PASS = True


def drop_first_upramp(d):
    """Return `d` with the initial rising ramp (the "first pass") removed.

    piezo_triangle_scan.py now records `points_per_ramp` in the CSV, so when that
    column is present we drop exactly the first ramp's worth of rows. For older
    files that predate it, the ramp length is inferred from the voltage-setpoint
    triangle: the first up-ramp ends at the first peak setpoint, so the index of
    the first maximum is the ramp length. The peak sample onward (the down-ramp)
    is kept. No-ops when the toggle is off or there's no full rising ramp.
    """
    if not OMIT_FIRST_PASS or len(d) < 2:
        return d
    if "points_per_ramp" in d.columns and d["points_per_ramp"].notna().all():
        ppr = int(d["points_per_ramp"].iloc[0])
    else:
        # first up-ramp ends where the setpoint first stops rising (its first
        # peak); argmax gives the first index of the max value = ramp length
        ppr = int(np.argmax(d["voltage_setpoint_V"].to_numpy()))
    if ppr <= 0 or ppr >= len(d):   # never rose / interrupted mid-ramp: keep as-is
        return d
    return d.iloc[ppr:].reset_index(drop=True)


# pick the most recent scan CSV by default; override CSV_PATH to load a specific file
# (non-recursive glob: master sub-scans live in scans/master_*/ and are handled
# separately in the "Master run" section below, so they don't collide here)
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "piezo_scan_*.csv")))
CSV_PATH = csv_files[-1] if csv_files else None
# CSV_PATH = "scans/piezo_scan_20260625_164729_min0.0_max40.0_axisX.csv"
print("Available scans:", [os.path.basename(f) for f in csv_files])
print("Loading:", CSV_PATH)

# which piezo axis was driven is encoded in the filename as "_axisX" / "_axisY"
# (added by piezo_triangle_scan.py). Fall back to Y for older files without it.
_m = re.search(r"_axis([XY])", os.path.basename(CSV_PATH or ""), re.IGNORECASE)
AXIS = _m.group(1).upper() if _m else "Y"
if _m is None:
    print("No axis token in filename; defaulting to AXIS =", AXIS)
else:
    print("Scanned axis (from filename):", AXIS)

# The camera is mounted rotated 90 deg, so a given piezo (physical) axis shows
# up on the *other* camera axis: piezo X -> camera Y, piezo Y -> camera X.
# We report results per piezo axis, so we keep the piezo-axis letter as the
# label everywhere but plot the camera column that actually carries that axis's
# motion.
CAM_OF_PIEZO = {"X": "Y", "Y": "X"}

# Mirror tilt angle (urad, converted from the Gaussian-fit center displacement)
# of the camera column that corresponds to the driven piezo axis.
POS_COL = f"gaussfit_center_{CAM_OF_PIEZO[AXIS].lower()}_urad"

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=["timestamp"])
df = add_mirror_angle_cols(df)   # add mirror-angle (urad) columns beside the _um ones
# optionally drop the first up-ramp (see OMIT_FIRST_PASS toggle above)
n_before = len(df)
df = drop_first_upramp(df)
if OMIT_FIRST_PASS:
    print(f"OMIT_FIRST_PASS on: dropped first up-ramp ({n_before - len(df)} of {n_before} samples)")
df.head()

## Voltage and mirror tilt angle vs. sample

In [ ]:
fig, ax_v = plt.subplots(figsize=(9, 4))
ax_v.plot(df["sample"], df["voltage_readback_V"], color="tab:blue", label="Piezo voltage (readback)")
ax_v.set_xlabel("Sample")
ax_v.set_ylabel("Voltage (V)", color="tab:blue")
ax_v.tick_params(axis="y", labelcolor="tab:blue")

ax_p = ax_v.twinx()
# label each trace by the piezo axis it represents: camera X carries piezo Y and
# camera Y carries piezo X (camera is rotated 90 deg)
ax_p.plot(df["sample"], df["gaussfit_center_x_urad"], color="tab:red", label=f"Piezo {CAM_OF_PIEZO['X']}")
ax_p.plot(df["sample"], df["gaussfit_center_y_urad"], color="tab:orange", label=f"Piezo {CAM_OF_PIEZO['Y']}")
ax_p.set_ylabel("Mirror tilt angle (urad)", color="tab:red")
ax_p.tick_params(axis="y", labelcolor="tab:red")
ax_p.legend(loc="upper right")

fig.suptitle("Piezo voltage and mirror tilt angle vs. sample")
fig.tight_layout()
plt.show()

## Driven-axis mirror angle vs. voltage

Each point is colored by sample index (time order), so the rising and
falling legs of the triangle wave are distinguishable - a gap between
the two colors at the same voltage indicates piezo hysteresis. The plotted
quantity is the mirror tilt angle (converted from the Gaussian-fit center
displacement) of the camera column that carries the driven piezo axis's motion
(camera rotated 90 deg), labeled by the piezo axis.

In [ ]:
elapsed_s = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(df["voltage_readback_V"], df[POS_COL],
                 c=elapsed_s, cmap="viridis", s=20)
ax.plot(df["voltage_readback_V"], df[POS_COL], "-", color="gray", alpha=0.3, linewidth=0.8)

ax.set_xlabel("Piezo voltage, readback (V)")
ax.set_ylabel(f"Mirror {AXIS} tilt angle (urad)")
ax.set_title(f"Mirror {AXIS} tilt angle vs. piezo voltage")

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Time (s)")

# max spread in mirror angle among samples that share the same voltage setpoint --
# this is the hysteresis gap between the rising and falling legs of the triangle wave
spread_by_voltage = df.groupby("voltage_setpoint_V")[POS_COL].agg(lambda s: s.max() - s.min())
max_spread = spread_by_voltage.max()
max_spread_voltage = spread_by_voltage.idxmax()

ax.annotate(
    f"Max angle spread\nat same voltage: {max_spread:.2f} urad\n(at {max_spread_voltage:.1f} V)",
    xy=(0.02, 0.02), xycoords="axes fraction", va="bottom", ha="left",
    fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.8),
)

fig.tight_layout()
plt.show()

print(f"Max mirror-angle spread at the same voltage: {max_spread:.3f} urad, at setpoint {max_spread_voltage:.2f} V")

## Linear fit: driven-axis sensitivity (urad/V)

In [ ]:
slope, intercept = np.polyfit(df["voltage_readback_V"], df[POS_COL], 1)
print(f"Piezo {AXIS} sensitivity: {slope:.3f} urad/V")
print(f"Intercept: {intercept:.3f} urad")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["voltage_readback_V"], df[POS_COL], s=15, alpha=0.6, label="Data")

v_fit = np.linspace(df["voltage_readback_V"].min(), df["voltage_readback_V"].max(), 200)
ax.plot(v_fit, slope * v_fit + intercept, color="tab:red",
        label=f"Linear fit: {slope:.3f} urad/V")

ax.set_xlabel("Piezo voltage, readback (V)")
ax.set_ylabel(f"Mirror {AXIS} tilt angle (urad)")
ax.set_title(f"Mirror {AXIS} tilt angle vs. voltage with linear fit")
ax.legend()
fig.tight_layout()
plt.show()

# Master run — per-subscan analysis

Analyzes a whole `piezo_master_scan.py` session at once. A master run lives in
`scans/master_<timestamp>/` and holds one triangle sub-scan CSV per
(piezo axis, setpoint). This section:

1. picks the most recent master run (override `MASTER_DIR` to choose another),
   parses each sub-scan's axis + setpoint from its filename, and builds a
   summary table of **sensitivity (urad/V)** and **max hysteresis (urad)** per
   sub-scan;
2. plots the **hysteresis loop for every sub-scan** (mirror angle vs. voltage,
   colored by time order); and
3. plots the headline **mirror-angle-per-volt and hysteresis vs. setpoint**,
   one series per axis.

All plots use the mirror tilt angle (converted from the Gaussian-fit center
displacement via `DETECTOR_DISTANCE`) in the piezo frame (same rotation handling
as the single-scan section: piezo X → camera Y, piezo Y → camera X).

In [ ]:
# --- select a master run and load its sub-scans ---
# each master run (piezo_master_scan.py) writes one sub-scan CSV per
# (axis, setpoint) into scans/master_<timestamp>/.
master_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "master_*")))
# MASTER_DIR = master_dirs[-1] if master_dirs else None
MASTER_DIR = "scans/master_20260702_165331"   # override to pick a specific run
print("Master runs:", [os.path.basename(d) for d in master_dirs])
print("Using:", MASTER_DIR)

_SUB_RE = re.compile(r"min([\d.]+)_max([\d.]+)_axis([XY])", re.IGNORECASE)


def load_subscan(path):
    """Return (meta dict, dataframe) for one sub-scan CSV.

    meta carries the driven piezo axis, the setpoint center (mean of the
    filename's min/max), and pos_col -- the mirror-angle (urad) column for the
    camera axis that carries that piezo axis's motion (camera rotated 90 deg, so
    piezo X -> camera Y etc.). The first up-ramp is dropped here when the
    OMIT_FIRST_PASS toggle is on.
    """
    m = _SUB_RE.search(os.path.basename(path))
    v_min = float(m.group(1)) if m else np.nan
    v_max = float(m.group(2)) if m else np.nan
    axis = m.group(3).upper() if m else "Y"
    d = pd.read_csv(path, parse_dates=["timestamp"])
    d = add_mirror_angle_cols(d)   # add mirror-angle (urad) columns beside the _um ones
    d = drop_first_upramp(d)   # honours the OMIT_FIRST_PASS toggle
    meta = {
        "path": path,
        "axis": axis,
        "center": (v_min + v_max) / 2.0,
        "v_min": v_min,
        "v_max": v_max,
        "pos_col": f"gaussfit_center_{CAM_OF_PIEZO[axis].lower()}_urad",
    }
    return meta, d


subscan_files = sorted(glob.glob(os.path.join(MASTER_DIR, "*.csv"))) if MASTER_DIR else []
print(f"{len(subscan_files)} sub-scans found")

records = []
subscans = []   # list of (meta, df), reused by the plots below
for path in subscan_files:
    meta, d = load_subscan(path)
    pos = d[meta["pos_col"]]
    # sensitivity: slope of mirror angle vs. readback voltage (urad/V)
    slope, _ = np.polyfit(d["voltage_readback_V"], pos, 1)
    # hysteresis: max spread in mirror angle among samples at the same setpoint
    spread = d.groupby("voltage_setpoint_V")[meta["pos_col"]].agg(lambda s: s.max() - s.min())
    records.append({
        "axis": meta["axis"],
        "center_V": meta["center"],
        "sensitivity_urad_per_V": slope,
        "hysteresis_urad": spread.max(),
    })
    subscans.append((meta, d))

summary = (pd.DataFrame(records)
           .sort_values(["axis", "center_V"])
           .reset_index(drop=True)) if records else pd.DataFrame(
    columns=["axis", "center_V", "sensitivity_urad_per_V", "hysteresis_urad"])
summary

### One-time axis swap (this run only)

The piezo X/Y drive lines were physically swapped for the master run currently
loaded above, so the axis parsed from each sub-scan's filename is inverted. The
cell below relabels every sub-scan to the correct axis and repoints its plotted
camera column, then rebuilds `summary`. Run it **once** after the load cell, then
re-run the per-subscan and summary plot cells below. For a normally-wired run,
skip (or delete) this cell.

In [ ]:
# --- ONE-TIME: swap piezo X<->Y for the currently loaded master run ---
# The piezo X/Y drive lines were physically swapped for THIS run, so every
# sub-scan's filename axis (and hence the camera column plotted) is inverted.
# Re-label each sub-scan to the other axis, recompute its position column
# accordingly, and rebuild `summary`. Then re-run the plot cells below.
#
# This edits `subscans`/`summary` in place, so re-running it toggles the swap
# back -- run it exactly once per (re)load of the master run above. Skip/delete
# it for normally-wired runs.
_SWAP = {"X": "Y", "Y": "X"}
for meta, _d in subscans:
    meta["axis"] = _SWAP[meta["axis"]]
    meta["pos_col"] = f"gaussfit_center_{CAM_OF_PIEZO[meta['axis']].lower()}_urad"

records = []
for meta, d in subscans:
    pos = d[meta["pos_col"]]
    slope, _ = np.polyfit(d["voltage_readback_V"], pos, 1)
    spread = d.groupby("voltage_setpoint_V")[meta["pos_col"]].agg(lambda s: s.max() - s.min())
    records.append({
        "axis": meta["axis"],
        "center_V": meta["center"],
        "sensitivity_urad_per_V": slope,
        "hysteresis_urad": spread.max(),
    })
summary = (pd.DataFrame(records)
           .sort_values(["axis", "center_V"])
           .reset_index(drop=True))
print("Swapped piezo X<->Y for this run. Rebuilt summary:")
summary

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Per-subscan hysteresis: one subplot per sub-scan, mirror angle vs. voltage
# colored by normalized scan progress (0=start -> 1=end) so the rising/falling
# legs (hysteresis loop) are visible and a single colorbar is shared by the grid.
if subscans:
    n = len(subscans)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows),
                             squeeze=False)
    axes_flat = axes.ravel()
    norm = Normalize(0, 1)
    for ax, (meta, d) in zip(axes_flat, subscans):
        progress = np.linspace(0, 1, len(d))  # time order, normalized per sub-scan
        ax.scatter(d["voltage_readback_V"], d[meta["pos_col"]], c=progress,
                   cmap="viridis", norm=norm, s=12)
        ax.plot(d["voltage_readback_V"], d[meta["pos_col"]], "-", color="gray",
                alpha=0.3, linewidth=0.7)
        ax.set_title(f"Piezo {meta['axis']} @ {meta['center']:.1f} V", fontsize=9)
        ax.set_xlabel("V readback")
        ax.set_ylabel(f"Mirror {meta['axis']} angle (urad)")
    # blank any unused subplots in the grid
    for ax in axes_flat[n:]:
        ax.axis("off")

    sm = ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes_flat.tolist())
    cbar.set_label("Scan progress (start → end)")

    fig.suptitle(f"Per-subscan hysteresis ({os.path.basename(MASTER_DIR)})")
    plt.show()
else:
    print("No sub-scans to plot.")

In [ ]:
# Headline result of a master run: sensitivity (mirror angle per volt) and
# hysteresis as a function of operating setpoint, one series per piezo axis.
if not summary.empty:
    fig, (ax_s, ax_h) = plt.subplots(1, 2, figsize=(12, 5))
    for axis, g in summary.groupby("axis"):
        ax_s.plot(g["center_V"], g["sensitivity_urad_per_V"], "o-", label=f"Piezo {axis}")
        ax_h.plot(g["center_V"], g["hysteresis_urad"], "o-", label=f"Piezo {axis}")

    ax_s.set_xlabel("Setpoint center (V)")
    ax_s.set_ylabel("Sensitivity (urad/V)")
    ax_s.set_title("Mirror angle per volt vs. setpoint")
    ax_s.legend()

    ax_h.set_xlabel("Setpoint center (V)")
    ax_h.set_ylabel("Max hysteresis (urad)")
    ax_h.set_title("Hysteresis vs. setpoint")
    ax_h.legend()

    fig.suptitle(f"Master run summary ({os.path.basename(MASTER_DIR)})")
    fig.tight_layout()
    plt.show()
else:
    print("No summary to plot.")